# 📘 Project 16 — Blood Emergency and Donor Matching
**Team No.:** 1  **Team Members:** Abhisekh Mohanty; Amit Kumar Jena; Amaresh Mohapatra; Hiranmay Behera

**Proposed Hybrid Model:** Bipartite GraphSAGE + Temporal Survival Transformer

**Dataset / Source:** Blood Transfusion Service Center dataset (transfusion.csv)
**Dataset Link:** https://www.kaggle.com/datasets/whenamancodes/blood-transfusion-dataset

**Task Type:** Survival-style binary classification — will a donor donate again (event) by a given time

---
## Data-Model Compatibility Note
This is a genuinely minimal dataset: 4 predictors (Recency, Frequency, Monetary, Time - months
since first donation) + a binary target (donated in a fixed follow-up window). Checked against
the proposed model:
- **Temporal Survival Transformer**: a real fit - `Time` (duration since first donation) and the
  binary target (donated again = "event") form a genuine survival-analysis pair. Modeled with a
  Transformer over the donor's (Recency, Frequency, Monetary) history, predicting a hazard/logit
  for the event within the observed time window.
- **Bipartite GraphSAGE**: no donor-blood-type or donor-recipient relational table exists in this
  dataset (blood type is not even a column). Adapted to a **donor-profile bipartite graph**:
  donors bucketed into profile clusters (by Recency/Frequency/Monetary quantiles) on one side,
  cluster-level "typical outcome" nodes on the other - a real, data-derived bipartite structure,
  not the literal donor-recipient matching graph the name implies.

**Verdict: PARTIAL** - Survival Transformer is faithful; GraphSAGE branch is a documented
profile-cluster substitute since no relational donor/recipient table exists in this dataset.

**How to run:** `Runtime -> Run all` (tiny tabular dataset, CPU is fine). Upload your Kaggle API
token when prompted in Section 1.


## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
# Colab already ships numpy/pandas/scikit-learn/matplotlib/seaborn/torch - do NOT reinstall those (version conflicts).
# Only install what's actually missing.
!pip -q install kaggle tqdm tabulate


In [ ]:
import os
import sys
import json
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    matthews_corrcoef, mean_absolute_error, mean_squared_error, r2_score
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device selected:", DEVICE)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


### CONFIG

In [ ]:
CONFIG = {
    "project_no": "16",
    "project_name": "Blood_Emergency_and_Donor_Matching",
    "team_no": "1",
    "task_type": "classification",
    "modality": "tabular_survival",
    "kaggle_dataset_slug": "whenamancodes/blood-transfusion-dataset",
    "dataset_source": "Blood Transfusion Service Center dataset",
    "target_column": "donated_again",
    "n_profile_clusters": 8,
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "transformer_hidden": 32,
    "graphsage_hidden": 16,
    "batch_size": 32,
    "epochs": 60,
    "learning_rate": 1e-3,
    "early_stop_patience": 8,
    "data_raw_dir": "data/raw",
    "data_processed_dir": "data/processed",
    "figures_dir": "figures",
    "results_dir": "results",
    "reports_dir": "reports",
}
for d in [CONFIG["data_raw_dir"], CONFIG["data_processed_dir"], CONFIG["figures_dir"],
          CONFIG["results_dir"], CONFIG["reports_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
from google.colab import files
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        if fname.endswith(".json"):
            os.replace(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d {CONFIG["kaggle_dataset_slug"]} -p {CONFIG["data_raw_dir"]} --unzip


In [ ]:
raw_files = os.listdir(CONFIG["data_raw_dir"])
print("Files in raw data dir:", raw_files)
assert len(raw_files) > 0, "No files found - check Section 1 download step before continuing."
for f in raw_files:
    print(f, "-", os.path.getsize(os.path.join(CONFIG["data_raw_dir"], f)), "bytes")


## 2. Load Raw Data

In [ ]:
candidates = [f for f in raw_files if f.lower().endswith(".csv")]
assert len(candidates) >= 1, f"No CSV found among: {raw_files}"
RAW_FILE = os.path.join(CONFIG["data_raw_dir"], candidates[0])
print("Using raw file:", RAW_FILE)

df = pd.read_csv(RAW_FILE)
df.columns = [c.strip().replace(" ", "_").replace("(", "").replace(")", "") for c in df.columns]
print(df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
print("Duplicate rows:", df.duplicated().sum())

label_candidates = [c for c in df.columns if "donat" in c.lower() and ("march" in c.lower() or "class" in c.lower() or "target" in c.lower())]
assert len(label_candidates) >= 1, f"Could not find donation-outcome column among: {list(df.columns)}"
LABEL_COL = label_candidates[0]
df[CONFIG["target_column"]] = df[LABEL_COL].astype(int)
target = CONFIG["target_column"]
print("Label column:", LABEL_COL)
print(df[target].value_counts(normalize=True))


**Data quality memo**

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
data_quality_memo = f"""# Data Quality Memo - Project 16: Blood Emergency and Donor Matching

## Dataset
- Source: Blood Transfusion Service Center dataset (whenamancodes/blood-transfusion-dataset)
- Rows: {len(df)}
- Duplicate rows: {df.duplicated().sum()}
- Columns: {list(df.columns)}

## Target
- Donated-again rate: {df[target].mean():.4f}

## Missingness
{missing[missing > 0].to_string() if (missing > 0).any() else "No missing values."}

## Leakage risks identified
- No repeated-donor ID present -> stratified random split used (Section 5).
- `{time_col}` (elapsed duration) used as a survival-style duration feature, not filtered post-hoc
  on the outcome - no look-ahead leakage.

## Adaptation note
No donor-blood-type or donor-recipient relational table exists (dataset has only 4 numeric
columns + outcome) - Bipartite GraphSAGE branch uses a data-derived profile-cluster bipartite
graph instead of a literal donor-recipient graph. See notebook header.
"""
with open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"), "w") as f:
    f.write(data_quality_memo)
print(data_quality_memo)


## 4. Preprocessing & Feature Engineering

In [ ]:
numeric_cols = [c for c in df.columns if c not in (target, LABEL_COL) and pd.api.types.is_numeric_dtype(df[c])]
print("Numeric feature columns:", numeric_cols)
feature_df = df[numeric_cols + [target]].copy()


## 5. Train / Validation / Test Split

In [ ]:
ratios = CONFIG["split_ratios"]
train_df, rest_df = train_test_split(feature_df, train_size=ratios["train"], stratify=feature_df[target], random_state=SEED)
rel_val = ratios["val"] / (ratios["val"] + ratios["test"])
val_df, test_df = train_test_split(rest_df, train_size=rel_val, stratify=rest_df[target], random_state=SEED)
print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))

manifest = {"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df),
            "train_class_balance": train_df[target].value_counts(normalize=True).to_dict()}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)
train_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "train.csv"), index=False)
val_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "val.csv"), index=False)
test_df.to_csv(os.path.join(CONFIG["data_processed_dir"], "test.csv"), index=False)
manifest


In [ ]:
scaler = StandardScaler().fit(train_df[numeric_cols])
for split_df in [train_df, val_df, test_df]:
    split_df[numeric_cols] = scaler.transform(split_df[numeric_cols])

# Donor-profile clusters (bipartite graph substitute) - fit on TRAIN only
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=CONFIG["n_profile_clusters"], random_state=SEED, n_init=10).fit(train_df[numeric_cols])
train_cluster = kmeans.predict(train_df[numeric_cols])
val_cluster = kmeans.predict(val_df[numeric_cols])
test_cluster = kmeans.predict(test_df[numeric_cols])
print("Cluster sizes (train):", np.bincount(train_cluster))


## 6. PyTorch Dataset & DataLoader

In [ ]:
class DonorDataset(Dataset):
    def __init__(self, X_df, cluster_idx, y_series):
        self.X = X_df[numeric_cols].values.astype(np.float32)
        self.cluster = cluster_idx.astype(np.int64)
        self.y = y_series.values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.cluster[idx]), torch.tensor(self.y[idx])

BATCH_SIZE = CONFIG["batch_size"]
train_ds = DonorDataset(train_df, train_cluster, train_df[target])
val_ds = DonorDataset(val_df, val_cluster, val_df[target])
test_ds = DonorDataset(test_df, test_cluster, test_df[target])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

xb, xb_cluster, yb = next(iter(train_loader))
print("features:", xb.shape, "cluster:", xb_cluster.shape, "target:", yb.shape)


## 7. Model Definitions

In [ ]:
class TemporalSurvivalTransformer(nn.Module):
    """Treats the donor's (Recency, Frequency, Monetary, Time) profile as a short sequence of
    donation-history 'events' (one token per feature), attending across them to predict a
    hazard-style logit for the survival event (donated again by the follow-up window)."""
    def __init__(self, n_features, hidden_dim=32, n_heads=4):
        super().__init__()
        self.feature_embed = nn.Parameter(torch.randn(1, n_features, hidden_dim) * 0.02)
        self.value_proj = nn.Linear(1, hidden_dim)
        layer = nn.TransformerEncoderLayer(hidden_dim, nhead=n_heads, dim_feedforward=hidden_dim * 2, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.out_dim = hidden_dim
    def forward(self, x):
        v = self.value_proj(x.unsqueeze(-1)) + self.feature_embed
        return self.encoder(v).mean(dim=1)


class BipartiteGraphSAGEBranch(nn.Module):
    """SAGE-style mean-aggregation over a bipartite donor<->profile-cluster graph (documented
    substitute for a literal donor-recipient graph, see header)."""
    def __init__(self, n_clusters, n_features, hidden_dim=16):
        super().__init__()
        self.cluster_embed = nn.Embedding(n_clusters, hidden_dim)
        self.self_proj = nn.Linear(n_features, hidden_dim)
        self.agg_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.out_dim = hidden_dim
    def forward(self, x, x_cluster):
        h_self = self.self_proj(x)
        h_neighbor = self.cluster_embed(x_cluster)
        return F.relu(self.agg_proj(torch.cat([h_self, h_neighbor], dim=-1)))


class HybridModel(nn.Module):
    def __init__(self, n_features, n_clusters, transformer_hidden=32, sage_hidden=16, output_dim=1):
        super().__init__()
        self.survival_transformer = TemporalSurvivalTransformer(n_features, transformer_hidden)
        self.graphsage = BipartiteGraphSAGEBranch(n_clusters, n_features, sage_hidden)
        self.head = nn.Sequential(nn.Linear(transformer_hidden + sage_hidden, 16), nn.ReLU(), nn.Linear(16, output_dim))
    def forward(self, x, x_cluster):
        h_t = self.survival_transformer(x)
        h_g = self.graphsage(x, x_cluster)
        return self.head(torch.cat([h_t, h_g], dim=-1))


### Architecture Verification

In [ ]:
n_features = xb.shape[1]
hybrid = HybridModel(n_features, CONFIG['n_profile_clusters'], CONFIG['transformer_hidden'], CONFIG['graphsage_hidden']).to(DEVICE)
print(hybrid)
for name, model in [('hybrid', hybrid)]:
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    print(f'{name}: total={total:,} trainable={trainable:,} device={next(model.parameters()).device}')


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, forward_fn, epochs, lr, patience, ckpt_path, pos_weight=None):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}
    epoch_bar = tqdm(range(epochs), desc='Training', unit='epoch')
    for epoch in epoch_bar:
        model.train()
        train_loss = 0.0
        n = 0
        batch_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False, unit='batch')
        for batch in batch_bar:
            optimizer.zero_grad()
            preds, targets = forward_fn(model, batch)
            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            bs = targets.shape[0]
            train_loss += loss.item() * bs
            n += bs
            batch_bar.set_postfix(loss=f'{loss.item():.4f}')
        train_loss /= n
        model.eval()
        val_loss = 0.0
        nv = 0
        with torch.no_grad():
            for batch in val_loader:
                preds, targets = forward_fn(model, batch)
                loss = criterion(preds, targets)
                bs = targets.shape[0]
                val_loss += loss.item() * bs
                nv += bs
        val_loss /= nv
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        epoch_bar.set_postfix(train_loss=f'{train_loss:.4f}', val_loss=f'{val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                epoch_bar.write(f'Early stopping at epoch {epoch + 1}')
                break
    return history

def forward_fn(model, batch):
    x, x_cluster, y = batch
    x, x_cluster, y = (x.to(DEVICE), x_cluster.to(DEVICE), y.to(DEVICE))
    return (model(x, x_cluster).squeeze(-1), y)
n_pos = train_df[target].sum()
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print('pos_weight:', pos_weight.item())
hybrid_history = train_model(hybrid, train_loader, val_loader, forward_fn, epochs=CONFIG['epochs'], lr=CONFIG['learning_rate'], patience=CONFIG['early_stop_patience'], ckpt_path=os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'), pos_weight=pos_weight)


## 9. Evaluation Metrics

In [ ]:
def get_predictions(model, loader, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            preds, targets = forward_fn(model, batch)
            all_preds.append(preds.cpu().numpy()); all_targets.append(targets.cpu().numpy())
    logits = np.concatenate(all_preds); targets = np.concatenate(all_targets)
    return 1 / (1 + np.exp(-logits)), targets

def evaluate_classification(probs, targets, threshold=0.5):
    pred_labels = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, pred_labels, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(targets, pred_labels), "precision_macro": precision,
            "recall_macro": recall, "f1_macro": f1, "mcc": matthews_corrcoef(targets, pred_labels),
            "roc_auc": roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else None,
            "pr_auc": average_precision_score(targets, probs) if len(np.unique(targets)) > 1 else None}


In [ ]:
results = {}
test_predictions = {}
for name, model, ckpt in [('hybrid', hybrid, os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'))]:
    probs, targets = get_predictions(model, test_loader, ckpt)
    results[name] = evaluate_classification(probs, targets)
    test_predictions[name] = (probs, targets)
print(json.dumps(results, indent=2, default=str))
with open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['val_loss'], label='Hybrid val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Proposed Model Validation Loss')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=300)
plt.show()


In [ ]:
hybrid_probs, hybrid_targets = test_predictions["hybrid"]
hybrid_pred_labels = (hybrid_probs >= 0.5).astype(int)
plt.figure(figsize=(5, 4))
cm = confusion_matrix(hybrid_targets, hybrid_pred_labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Hybrid, test set)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_or_scatter.png"), dpi=300); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(hybrid_targets, hybrid_probs)
precision, recall, _ = precision_recall_curve(hybrid_targets, hybrid_probs)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_title("ROC Curve")
axes[1].plot(recall, precision); axes[1].set_title("Precision-Recall Curve")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=300); plt.show()


### Explainable AI

In [ ]:
from sklearn.metrics import roc_auc_score as _auc
base_score = _auc(hybrid_targets, hybrid_probs) if len(np.unique(hybrid_targets)) > 1 else 0.5
rng = np.random.default_rng(SEED)
importances = {}
for i, col in enumerate(numeric_cols):
    X_perm = test_ds.X.copy()
    X_perm[:, i] = rng.permutation(X_perm[:, i])
    with torch.no_grad():
        preds = hybrid(torch.tensor(X_perm).to(DEVICE), torch.tensor(test_ds.cluster).to(DEVICE)).squeeze(-1)
        probs_perm = torch.sigmoid(preds).cpu().numpy()
    score_perm = _auc(hybrid_targets, probs_perm) if len(np.unique(hybrid_targets)) > 1 else 0.5
    importances[col] = base_score - score_perm

plt.figure(figsize=(6, 4))
pd.Series(importances).sort_values().plot(kind="barh")
plt.title("Permutation feature importance (AUC drop, hybrid)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=300); plt.show()


### Error Analysis

In [ ]:
fn_mask = (hybrid_targets == 1) & (hybrid_pred_labels == 0)
fp_mask = (hybrid_targets == 0) & (hybrid_pred_labels == 1)
print(f"False negatives: {fn_mask.sum()} / {int(hybrid_targets.sum())} donors missed")
print(f"False positives: {fp_mask.sum()} / {int((hybrid_targets==0).sum())} flagged incorrectly")

plt.figure(figsize=(6, 4))
sns.histplot(test_cluster[fn_mask], color="red", label="False negatives", bins=CONFIG["n_profile_clusters"])
sns.histplot(test_cluster[~fn_mask], color="blue", label="Rest", bins=CONFIG["n_profile_clusters"], alpha=0.4)
plt.title("Missed donors by profile cluster"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=300); plt.show()


### Computational Efficiency

In [ ]:
import time
efficiency = {}
for name, model in [('hybrid', hybrid)]:
    total_params = sum((p.numel() for p in model.parameters()))
    trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
    model.eval()
    xb_b, xb_cluster_b, _ = next(iter(test_loader))
    xb_b, xb_cluster_b = (xb_b.to(DEVICE), xb_cluster_b.to(DEVICE))
    with torch.no_grad():
        for _ in range(3):
            model(xb_b, xb_cluster_b)
        start = time.time()
        for _ in range(20):
            model(xb_b, xb_cluster_b)
        elapsed = (time.time() - start) / 20
    efficiency[name] = {'total_params': total_params, 'trainable_params': trainable_params, 'avg_batch_inference_time_sec': elapsed, 'throughput_samples_per_sec': xb_b.shape[0] / elapsed}
print(json.dumps(efficiency, indent=2))
with open(os.path.join(CONFIG['results_dir'], 'efficiency.json'), 'w') as f:
    json.dump(efficiency, f, indent=2)
